# Can room occupancy be predicted?

This notebook treats `Occupancy` as a binary classification target and evaluates whether contemporaneous environmental sensor readings can predict it. Because observations are ordered by `date`, all model selection uses chronological splits—never random shuffling—so future observations cannot leak into training.

**Protocol:** first 60% for training, next 20% for validation/model selection, and final 20% for a one-time test. We compare a majority-class baseline with logistic regression, random forest, and histogram gradient boosting. Balanced accuracy is the primary selection metric because the target is imbalanced.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    classification_report, f1_score, precision_score, recall_score,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

In [ ]:
DATA_PATH = Path('data.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['date'])

required = {'date', 'Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio', 'Occupancy'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
assert df['date'].is_monotonic_increasing, 'Rows must be ordered by date.'
assert set(df['Occupancy'].dropna().unique()).issubset({0, 1})

summary = pd.Series({
    'rows': len(df),
    'start': df['date'].min(),
    'end': df['date'].max(),
    'duplicate timestamps': int(df['date'].duplicated().sum()),
    'missing values': int(df.isna().sum().sum()),
    'occupied share': df['Occupancy'].mean(),
})
display(summary.to_frame('value'))
display(df.head())

In [ ]:
sensor_features = ['Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df['Occupancy'].value_counts().sort_index().rename(index={0: 'vacant', 1: 'occupied'}).plot.bar(
    ax=axes[0], color=['#7f8c8d', '#2a9d8f'], rot=0
)
axes[0].set(title='Target class balance', ylabel='Rows', xlabel='')

corr = df[sensor_features + ['Occupancy']].corr(numeric_only=True)
sns.heatmap(corr, cmap='vlag', center=0, annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Feature correlations')
plt.tight_layout()
plt.show()

display(df.groupby('Occupancy')[sensor_features].median().rename(index={0: 'vacant', 1: 'occupied'}))

## Chronological holdout design

The split below simulates deployment: each model learns only from earlier rows and is evaluated on later rows. The timestamp is retained for auditing but is not used as a predictor. This asks whether the sensors generalize forward in time rather than whether the model can memorize dates.

In [ ]:
n = len(df)
train_end, valid_end = int(0.60 * n), int(0.80 * n)
train = df.iloc[:train_end].copy()
valid = df.iloc[train_end:valid_end].copy()
test = df.iloc[valid_end:].copy()

assert train['date'].max() < valid['date'].min() < test['date'].min()

split_summary = pd.DataFrame({
    name: {
        'rows': len(part),
        'start': part['date'].min(),
        'end': part['date'].max(),
        'occupied_share': part['Occupancy'].mean(),
    }
    for name, part in [('train', train), ('validation', valid), ('test', test)]
}).T
display(split_summary)

In [ ]:
X_train, y_train = train[sensor_features], train['Occupancy']
X_valid, y_valid = valid[sensor_features], valid['Occupancy']
X_test, y_test = test[sensor_features], test['Occupancy']

models = {
    'majority baseline': DummyClassifier(strategy='most_frequent'),
    'logistic regression': make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    ),
    'random forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    'histogram gradient boosting': HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=200, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    ),
}

def metric_row(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        'balanced_accuracy': balanced_accuracy_score(y_true, predictions),
        'accuracy': accuracy_score(y_true, predictions),
        'precision': precision_score(y_true, predictions, zero_division=0),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'f1': f1_score(y_true, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probabilities),
    }

fitted = {}
validation_rows = []
for name, model in models.items():
    fitted[name] = clone(model).fit(X_train, y_train)
    probabilities = fitted[name].predict_proba(X_valid)[:, 1]
    validation_rows.append({'model': name, **metric_row(y_valid, probabilities)})

validation_results = (pd.DataFrame(validation_rows)
                      .set_index('model')
                      .sort_values('balanced_accuracy', ascending=False))
display(validation_results.style.format('{:.3f}'))

## Select the model and operating threshold

We select the non-baseline model with the highest validation balanced accuracy. We then choose its probability threshold on validation data only. Threshold selection matters here because occupied rows are less common, and the default 0.50 threshold need not balance occupied and vacant recall.

In [ ]:
candidate_results = validation_results.drop(index='majority baseline')
selected_name = candidate_results['balanced_accuracy'].idxmax()
selected_model = fitted[selected_name]
valid_probability = selected_model.predict_proba(X_valid)[:, 1]

thresholds = np.linspace(0.05, 0.95, 181)
threshold_scores = np.array([
    balanced_accuracy_score(y_valid, valid_probability >= threshold)
    for threshold in thresholds
])
selected_threshold = float(thresholds[threshold_scores.argmax()])

print(f'Selected model: {selected_name}')
print(f'Selected threshold: {selected_threshold:.3f}')
print(f'Validation balanced accuracy: {threshold_scores.max():.3f}')

plt.figure(figsize=(8, 3.5))
plt.plot(thresholds, threshold_scores, color='#264653')
plt.axvline(selected_threshold, color='#e76f51', linestyle='--', label=f'selected = {selected_threshold:.2f}')
plt.xlabel('Probability threshold')
plt.ylabel('Validation balanced accuracy')
plt.title('Threshold selection on validation data')
plt.legend()
plt.show()

## Final evaluation on unseen future data

The final block has not been used for model or threshold selection. Performance here is the answer to the prediction question.

In [ ]:
test_probability = selected_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= selected_threshold).astype(int)
test_metrics = pd.Series(
    metric_row(y_test, test_probability, selected_threshold),
    name='test score',
)
display(test_metrics.to_frame().style.format('{:.3f}'))
print(classification_report(y_test, test_prediction, target_names=['vacant', 'occupied'], digits=3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, display_labels=['vacant', 'occupied'], cmap='Blues', ax=axes[0], colorbar=False
)
axes[0].set_title('Test confusion matrix')
roc_display = RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[1])
roc_display.line_.set_color('#2a9d8f')
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('Test ROC curve')
pr_display = PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[2])
pr_display.line_.set_color('#e76f51')
axes[2].set_title('Test precision–recall curve')
plt.tight_layout()
plt.show()

In [ ]:
importance = permutation_importance(
    selected_model, X_test, y_test, scoring='balanced_accuracy',
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
importance_table = (pd.DataFrame({
    'feature': sensor_features,
    'importance_mean': importance.importances_mean,
    'importance_std': importance.importances_std,
}).sort_values('importance_mean', ascending=False))
display(importance_table.style.format({'importance_mean': '{:.3f}', 'importance_std': '{:.3f}'}))

plt.figure(figsize=(8, 3.5))
plt.barh(importance_table['feature'][::-1], importance_table['importance_mean'][::-1],
         xerr=importance_table['importance_std'][::-1], color='#457b9d')
plt.xlabel('Decrease in balanced accuracy after permutation')
plt.title('Test-set permutation importance (descriptive only)')
plt.show()

In [ ]:
baseline_balanced = validation_results.loc['majority baseline', 'balanced_accuracy']
verdict = 'YES' if test_metrics['balanced_accuracy'] >= 0.80 and test_metrics['roc_auc'] >= 0.85 else 'PROMISING, BUT NOT YET CONCLUSIVE'
print(f'VERDICT: {verdict}')
print(f'The selected {selected_name} achieved test balanced accuracy '      f'{test_metrics["balanced_accuracy"]:.3f} and ROC AUC {test_metrics["roc_auc"]:.3f}.')
print(f'The majority baseline has balanced accuracy {baseline_balanced:.3f}; '      'the model must materially exceed 0.50 to demonstrate predictive signal.')
print('Interpretation is limited to this room, sensor setup, and time period. A stronger deployment test would use '      'a later collection period or another room, and monitor sensor drift and decision costs.')

## Interpretation and limitations

- A materially better-than-0.50 test balanced accuracy demonstrates forward-in-time predictive signal beyond always choosing the majority class.
- The model uses sensor readings measured at the same instant as occupancy. It supports occupancy detection, not advance forecasting. Advance prediction would require lagged features and a forecast horizon.
- The test set is temporally later but comes from the same room and short collection period. Results may not transfer to other rooms, seasons, sensor calibrations, or operating schedules.
- Permutation importance describes associations used by this fitted model; it does not establish causality.
- In deployment, choose the probability threshold based on the relative cost of false alarms versus missed occupancy, and retrain only after collecting labels from later periods.